In [1]:
INPUT_WB_NAME  = "2026 Inputs for Apps.xlsx"

####
INPUT_WS_NAME  = "SPOT INPUTS"
INPUT_TBL_NAME = "SPOT_INPUTS"
####

In [2]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

# --- autoreload ---
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from collections import defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output

In [4]:
# --- builders ---
from fin_insts import make_single_leg_fin_insts, BestOf #, FutureSpread, Synthetic

In [5]:
# --- IBKR ---
from ibkr.Class_IBKR_IB import IBKR_IB
# from ibkr.Class_IBKR_TWS import IBKR_TWS
ibkr = IBKR_IB()

In [6]:
# --- feeds ---
from ws_feeds import WSFeedManager

In [7]:
# --- output ---
from output.Output_Methods import create_output
from output.Class_xlWings import xlWings
xlw = xlWings()

In [8]:
# --- utils ---
# from other.Graph_Theory import find_all_node_permutations, connect_nodes_with_edges
from output.Output_Methods import create_output
from output.Class_xlWings import xlWings
xlw = xlWings()

In [9]:
# --- trading strategy ---
# from strategies import Strategy, TradePackage

In [10]:
# CONSTANTS

DB_WB_NAME  = "2026 Crypto Products Database.xlsx"

OUTPUT_COLS = [
               'time',
    
               'my_prod_type',
               'my_fi_name',
               'my_pf_name',
    
               'numerator_currency',
               'denominator_currency',
                              
               'price_mkt_bid',
               'price_mkt_ask',
    
               'scalar_price_mkt_to_unit',
    
               'price_unit_bid',
               'price_unit_ask'
            ]

In [11]:
async def standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME):

    df = xlw.get_df(INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME, table=True)
    input_dict = df.set_index('Keys')['Values'].to_dict()
    
    wb  = input_dict['input workbook name']
    ws  = input_dict['true/false sheet name']
    tbl = input_dict['true/false table name']
    true_false_df = xlw.get_df(wb, ws, tbl, table=True)
    
    if 'TRUE/FALSE' not in true_false_df.columns:
        true_false_df = true_false_df.set_index('Keys').T

    true_false_df = true_false_df[true_false_df['TRUE/FALSE'] == True]
    
    wb  = DB_WB_NAME
####    
    ws  = input_dict['crypto long name']
    tbl = input_dict['crypto abbrev'] + "_static_data_table"
####
    
    db_df = xlw.get_df(wb, ws, tbl, table=True)
    
    merged_df = true_false_df.merge(db_df,how='left',on=['my_fi_name', 'my_pf_name'])

    fin_inst_objs_list = make_single_leg_fin_insts(merged_df)

    return input_dict, fin_inst_objs_list

In [12]:
async def main():

    input_dict, objs_list = await standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME)

    ws_objs_list = [obj for obj in objs_list if obj.my_pf_name != 'IBKR']
    ws_feed      = WSFeedManager(ws_objs_list)

    await ws_feed.complete_fi_objects()   
        
    ibkr_objs_list     = [obj for obj in objs_list if obj.my_pf_name == 'IBKR']
    if ibkr_objs_list:
        await ibkr.connect()
        print("IBKR connected:", ibkr.ib.isConnected())
        
        await asyncio.gather(*(ibkr.create_simple_contract(obj) for obj in ibkr_objs_list))
        await asyncio.gather(*(ibkr.complete_obj(obj) for obj in ibkr_objs_list))
    
    ''' 
    # insert ibkr BAG instruments here (future_spread, option_spread, option_combo, etc.)
    futures_list = [obj for obj in ibkr_objs_list if obj.my_prod_type == 'future']
    bag_objs_list = FutureSpread.make_spreads(futures_list)                                    
    await asyncio.gather(*(ibkr.create_bag_contract(obj) for obj in bag_objs_list))
    '''                
        
    ''' 
    insert synthetic instruments here 
    syn_objs_list = 
    ''' 

    #''' 
    #insert bestOf instruments here
    bo_objs_dict = defaultdict(list)

    for obj in [*ws_objs_list, *ibkr_objs_list]:
        if obj.my_prod_type == 'spot':
            bo_objs_dict[obj.my_fi_name].append(obj)
        else:
            bo_objs_dict[obj.my_prod_type].append(obj)
            
    bo_objs_list = []
    attr_list = [('price_unit_bid', max), 
                 ('price_unit_ask', min)]
    for curr, obj_list in bo_objs_dict.items():
        if curr == 'equity':
            bo_obj = BestOf("ETF", obj_list, attr_list, mode='auto')
        else:
            bo_obj = BestOf(f"{curr}", obj_list, attr_list, mode='auto')
            
        bo_objs_list.append(bo_obj)
    
    bo_objs_list.sort(key=lambda obj: obj.my_fi_name)
    #''' 
  
    '''
    insert trading and analysis scripts here 
    strat_df = xlw.get_df(INPUT_WB_NAME, INPUT_WS_NAME, STRAT_TBL_NAME, table=True)

    for obj in ibkr_objs_list:
        obj.platform_obj = ibkr  # this is the object not the name
    '''
    
    output_list = [
        *bo_objs_list,
        *ws_objs_list,
        *ibkr_objs_list,
        #*bag_objs_list,
        #*syn_objs_list,
        #*strat_objs_list
                ]
    
    # Run all streams concurrently
    tasks = []
    tasks.append(asyncio.create_task(ws_feed.run()))
    tasks.append(asyncio.create_task(create_output(input_dict, output_list, OUTPUT_COLS)))
    if ibkr_objs_list:
        tasks.append(asyncio.create_task(ibkr.start_streams(ibkr_objs_list)))
        #tasks.append(asyncio.create_task(ibkr.start_streams(bag_objs_list)))

    #'''
    for obj in bo_objs_list:
        tasks.append(asyncio.create_task(obj.run_timer())) 

    await asyncio.sleep(15)

    '''
    await strat.done_event.wait()
    
    # then cancel everything else
    for task in tasks:
        task.cancel()
    
    # optional: wait for clean cancellation
    await asyncio.gather(*tasks, return_exceptions=True)
    
    # disconnect IBKR
    ibkr.ib.disconnect()
    
    print("Program finished cleanly.")
    '''

In [13]:
await main()

Next Order ID: 3
IBKR connected: True
2026-05-08_09-51-54
2026-05-08_09-52-25
2026-05-08_09-52-55
2026-05-08_09-53-25
2026-05-08_09-53-55
2026-05-08_09-54-25
2026-05-08_09-54-56
2026-05-08_09-55-26
2026-05-08_09-55-56
2026-05-08 09:56:12.217656 Gemini WS error (ConnectionClosedError): no close frame received or sent. Reconnecting in 2s...
2026-05-08_09-56-26
2026-05-08 09:56:27.183471 Gemini WS error (ConnectionClosedError): no close frame received or sent. Reconnecting in 2s...
2026-05-08_09-56-57
2026-05-08_09-57-27
2026-05-08_09-57-57
2026-05-08_09-58-27
2026-05-08_09-58-58
2026-05-08_09-59-28
2026-05-08_09-59-58
2026-05-08_10-00-28
2026-05-08_10-00-59
2026-05-08_10-01-29
2026-05-08_10-01-59
2026-05-08_10-02-29
2026-05-08_10-03-00
2026-05-08_10-03-30
2026-05-08_10-04-00
2026-05-08_10-04-30
2026-05-08_10-05-01
2026-05-08_10-05-31
2026-05-08_10-06-01
2026-05-08_10-06-31
2026-05-08_10-07-02
2026-05-08_10-07-32
2026-05-08_10-08-02
2026-05-08_10-08-33
2026-05-08_10-09-03
2026-05-08_10-09